# phase_4 (M4 T-KAN) on Kaggle — Drive-resumable

Per-region temporal progression (`improved/stable/worsened`) on top of a **frozen M3**.
Implements `docs/VERA_phase_3_4_5_spec.md` §4.

**Two stages in one notebook:**
1. **Precompute** (once) — freeze the trained M3 and cache region features+logits for every image
   (`phase_3/precompute_regions.py`). Needs the BioViL-T feature cache + the M3 checkpoint.
   Set `DO_PRECOMPUTE=1` the first time; afterwards upload the cache as a dataset and set `0`.
2. **Train M4** — reads the region cache only (no backbone).

**Inputs (Kaggle datasets):** `phase3-code`, `phase4-code`, `m3-labels`, `m4-labels`
(bundle `m3_pairs.jsonl` here), the M3 checkpoint, and either `mimic-biovilt-feats` (for
precompute) or a prebuilt `m3-region-cache`. Settings → GPU, Internet On, then Run All.

## CONFIG — edit these

In [ ]:
import os
PHASE3_SRC = '/kaggle/input/phase3-code/phase_3'
PHASE4_SRC = '/kaggle/input/phase4-code/phase_4'
M3_LABELS  = '/kaggle/input/m3-labels'
M4_LABELS  = '/kaggle/input/m4-labels'                 # also holds m3_pairs.jsonl
PAIRS      = '/kaggle/input/m4-labels/m3_pairs.jsonl'

# --- precompute stage ---
DO_PRECOMPUTE = 1                                       # 1 first time; 0 once you have a cache dataset
M3_CKPT    = '/kaggle/input/m3-ckpt/m3_A/best.pt'       # trained mode-A M3 (the shipping head)
FEATURES   = '/kaggle/input/mimic-biovilt-feats'       # BioViL-T cache (needed only to precompute)
REGION_CACHE = '/kaggle/working/m3_region_cache' if DO_PRECOMPUTE else '/kaggle/input/m3-region-cache'

# --- train stage ---
EPOCHS   = 40
RUN_NAME = 'm4'
RESUME   = 0
RCLONE_REMOTE = 'dhint:CHEX-DATA/m4_runs'
SYNC_EVERY    = 0
RCLONE_SECRET = 'RCLONE_CONF'

for k, v in dict(PHASE3_SRC=PHASE3_SRC, PHASE4_SRC=PHASE4_SRC, M3_LABELS=M3_LABELS, M4_LABELS=M4_LABELS,
                 PAIRS=PAIRS, DO_PRECOMPUTE=DO_PRECOMPUTE, M3_CKPT=M3_CKPT, FEATURES=FEATURES,
                 REGION_CACHE=REGION_CACHE, EPOCHS=EPOCHS, RUN_NAME=RUN_NAME, RESUME=RESUME,
                 RCLONE_REMOTE=RCLONE_REMOTE, SYNC_EVERY=SYNC_EVERY, RCLONE_SECRET=RCLONE_SECRET).items():
    os.environ[k] = str(v)
print('config set | precompute', DO_PRECOMPUTE, '| resume', RESUME)

## 1 — rclone + Drive (graceful: skips sync if no secret)
Copy your local `rclone.conf` text into a Kaggle Secret named `RCLONE_CONF`.

In [ ]:
%%bash
set -e
if ! command -v rclone >/dev/null 2>&1; then
  cd /kaggle/working && curl -sLO https://downloads.rclone.org/rclone-current-linux-amd64.zip
  unzip -q -o rclone-current-linux-amd64.zip && cp rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod +x /usr/local/bin/rclone
fi
rclone version | head -1

In [ ]:
import os, pathlib
SYNC_OK = '0'
try:
    from kaggle_secrets import UserSecretsClient
    conf = UserSecretsClient().get_secret(os.environ['RCLONE_SECRET'])
    cfg = pathlib.Path.home() / '.config' / 'rclone'; cfg.mkdir(parents=True, exist_ok=True)
    (cfg / 'rclone.conf').write_text(conf)
    if os.system('rclone mkdir "%s"' % os.environ['RCLONE_REMOTE']) == 0:
        SYNC_OK = '1'; print('remote OK ->', os.environ['RCLONE_REMOTE'])
    else:
        print('[WARN] remote unreachable -> training WITHOUT Drive sync')
except Exception as e:
    print('[WARN] no RCLONE_CONF secret -> training WITHOUT Drive sync:', e)
os.environ['SYNC_OK'] = SYNC_OK; print('SYNC_OK =', SYNC_OK)

## 2 — copy code (phase_3 for precompute + phase_4)

In [ ]:
!rm -rf /kaggle/working/phase_3 /kaggle/working/phase_4
!cp -r "$PHASE3_SRC" /kaggle/working/phase_3 && cp -r "$PHASE4_SRC" /kaggle/working/phase_4
!cd /kaggle/working/phase_4 && python -c "import constants as C; print('regions',C.NUM_REGIONS,'prog',C.PROG_NAMES)"

## 3 — precompute frozen-M3 region cache (only if DO_PRECOMPUTE=1)
Runs over every image with features (current + prior). ~6-7 GB; push to Drive to reuse, or
Save Version → download → make an `m3-region-cache` dataset and set `DO_PRECOMPUTE=0` next time.

In [ ]:
import os
if int(os.environ['DO_PRECOMPUTE']):
    !cd /kaggle/working/phase_3 && python precompute_regions.py --ckpt "$M3_CKPT" \
      --labels-dir "$M3_LABELS" --features-root "$FEATURES" --out-dir "$REGION_CACHE" --device cuda
    if os.environ.get('SYNC_OK') == '1':
        os.system('rclone copy "%s" "%s/m3_region_cache" --transfers 8 --quiet' % (os.environ['REGION_CACHE'], os.environ['RCLONE_REMOTE']))
        print('region cache pushed to Drive')
else:
    print('skip precompute; using', os.environ['REGION_CACHE'])

## 4 — train M4 (Drive-synced)
Re-run with `RESUME=1` after a session dies. Selection metric = val 3-class **macro-F1**
(change-only F1 reported alongside; accuracy≈stable is a red flag).

In [ ]:
import os
R = '--resume' if int(os.environ['RESUME']) else ''
S = ('--sync-remote "%s" --sync-every %s' % (os.environ['RCLONE_REMOTE'], os.environ['SYNC_EVERY'])) if os.environ.get('SYNC_OK') == '1' else ''
os.environ.update(R_FLAG=R, S_FLAG=S)
print('flags:', R, S or '(no sync)')
!cd /kaggle/working/phase_4 && python train.py --region-cache "$REGION_CACHE" \
  --m3-labels-dir "$M3_LABELS" --m4-labels-dir "$M4_LABELS" --pairs "$PAIRS" \
  --out /kaggle/working/m4_runs --name "$RUN_NAME" --epochs $EPOCHS --device cuda \
  --workers 4 $S_FLAG $R_FLAG

## 5 — eval (test) + infer (change-ledger JSON for M5)

In [ ]:
!cd /kaggle/working/phase_4 && python eval.py --ckpt /kaggle/working/m4_runs/$RUN_NAME/best.pt \
  --region-cache "$REGION_CACHE" --m3-labels-dir "$M3_LABELS" --m4-labels-dir "$M4_LABELS" \
  --pairs "$PAIRS" --split test --device cuda

In [ ]:
import os
!cd /kaggle/working/phase_4 && python infer.py --ckpt /kaggle/working/m4_runs/$RUN_NAME/best.pt \
  --region-cache "$REGION_CACHE" --m3-labels-dir "$M3_LABELS" --m4-labels-dir "$M4_LABELS" \
  --pairs "$PAIRS" --split test --out /kaggle/working/m4_pred_$RUN_NAME.jsonl --device cuda
if os.environ.get('SYNC_OK') == '1':
    os.system('rclone copy /kaggle/working/m4_runs/%s "%s/%s" --quiet' % (os.environ['RUN_NAME'], os.environ['RCLONE_REMOTE'], os.environ['RUN_NAME']))
    print('final run pushed to Drive')